<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 30px; border-radius: 15px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center; box-shadow: 0 10px 20px rgba(0,0,0,0.19), 0 6px 6px rgba(0,0,0,0.23);">
    <div style="font-size: 50px; margin-bottom: 10px;"> 🤖</div>
    <h1 style="margin: 0; font-size: 36px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
        Tox21: Double Deep Q-Network
    </h1>
    <p style="font-size: 18px; margin-top: 5px; font-weight: 300;">
        Molecular decision-making with Double Deep Q-Networks
    </p>
    <p style="font-size: 16px; margin-top: 10px; font-weight: 300; line-height: 1.5;">
    The notebook is structured as follows: <br>
     <strong>Imports</strong> |  <strong>Dataset</strong> |  <strong>GNN Model</strong> |  <strong>Training</strong> |  <strong>Hyperparameter Search</strong> |  <strong>Test</strong>
</p>
</div>

# **Imports**

In [3]:
import torch
import os
import numpy as np
import random
from sklearn.model_selection import train_test_split
from rdkit import Chem, RDLogger
from src.DDQN.env import *
from src.DDQN.train import train_agent, DuelingDQN
from src.DDQN.evaluate import evaluate_model
from src.DGN.model import Tox21GNN
from src.DDQN.config import *
from src.DDQN.analysis import analysis
import pandas as pd
from src.DDQN.utils import set_seed
from src.DGN.config import MODEL_PATH as GNN_PATH
from src.DDQN.config import AGENT_PATH
set_seed(RANDOM_SEED)
RDLogger.DisableLog('rdApp.*') # disable rdkit warnings

# **Dataset**

In [4]:
# ---LOAD CUSTOM DATASET Tox21
tox21_df=pd.read_csv("../datasets/tox21_processed_features.csv")
all_smiles=tox21_df["smiles"].values
print(f"DEBUG: Caricate {len(all_smiles)} righe dal CSV.")


train_smiles,temp_smiles = train_test_split(all_smiles, train_size=0.7, test_size=0.3, random_state=RANDOM_SEED)
valid_smiles, test_smiles = train_test_split(temp_smiles,test_size=0.5, random_state=RANDOM_SEED)

print(f"Totale molecole: {len(all_smiles)}")
print(f"Train Set (per RL Training): {len(train_smiles)} molecole")
print(f"Test Set (per Evaluation):   {len(test_smiles)} molecole")

DEBUG: Caricate 7831 righe dal CSV.
Totale molecole: 7831
Train Set (per RL Training): 5481 molecole
Test Set (per Evaluation):   1175 molecole


# **DGN** Predictor

In [5]:

print(f"Running on device: {DEVICE}")

print("Initializing GNN Predictor...")
gnn_model = Tox21GNN(num_node_features=NODE_FEATURES, 
                        hidden_channels=HIDDEN_CHANNELS, 
                        num_classes=NUM_CLASSES, 
                        dropout=DROPOUT,
                        num_global_features=NUM_GLOBAL_FEATURES)
gnn_model = gnn_model.to(DEVICE)

if os.path.exists(GNN_PATH):
    gnn_model.load_state_dict(torch.load(GNN_PATH, map_location=DEVICE))
    print(f"Uploaded DGN weights from {GNN_PATH}")
else:
    print(f"DGN weights not found at {GNN_PATH}!")
    
gnn_model.eval()


Running on device: cuda
Initializing GNN Predictor...
Uploaded DGN weights from weights/DGN/best_GNN.pth


Tox21GNN(
  (node_encoder): Linear(in_features=88, out_features=128, bias=True)
  (bond_encoder): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU()
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): SiLU()
  )
  (conv1): GINEConv(nn=MLP(
    (lin1): Linear(in_features=128, out_features=256, bias=True)
    (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (lin2): Linear(in_features=256, out_features=128, bias=True)
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): SiLU()
    (dropout): Dropout(p=0.41573689676626024, inplace=False)
  ))
  (conv2): GINEConv(nn=MLP(
    (lin1): Linear(in_features=128, out_features=256, bias=True)
    (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (lin2): Linear(in_features

# **Environment**

In [ ]:

print(f"Initializing Environment for: {train_smiles}")
env = MoleculeEnv(gnn_model=gnn_model,threshold=0.6,max_steps=MAX_STEPS,device=DEVICE,w_tox=W_TOX,w_flip=W_FLIP,w_sim_penalty=W_PEN)


# **DDQN** Agent

In [ ]:

policy_net = DuelingDQN(LATENT_DIM, env.action_space_size, hidden_dim=512)
target_net = DuelingDQN(LATENT_DIM, env.action_space_size, hidden_dim=512)


# **Train**

In [ ]:

print(f"Starting Training for {EPOCHS_AGENT} episodes...")

"""
trained_agent = train_agent(
    p_model=policy_net,
    t_model=target_net,
    env,
    smiles_list=train_smiles, 
    batch_size=BATCH_SIZE_RL_AGENT, 
    lr=LR_GENERATOR,
    gamma=GAMMA
)
"""

#torch.save(trained_agent.state_dict(), AGENT_PATH)
#print(f"Agent saved to {AGENT_PATH}")


# **Results**

In [ ]:

if os.path.exists(AGENT_PATH):
    print(f"Loading Best Agent from {AGENT_PATH} for evaluation...")
    best_agent = DuelingDQN(input_dim=LATENT_DIM, output_dim=env.action_space_size,hidden_dim=512).to(DEVICE)
    best_agent.load_state_dict(torch.load(AGENT_PATH))
    stats = evaluate_model(best_agent, env, test_smiles, device=DEVICE)
else:
    trained_agent = DuelingDQN(input_dim=LATENT_DIM, output_dim=env.action_space_size,hidden_dim=512).to(DEVICE)
    trained_agent.load_state_dict(torch.load(AGENT_PATH))

    stats=evaluate_model(trained_agent, env, test_smiles, device=DEVICE)
    

analysis(stats=stats)